In [1]:
"""
======================================================
PROJECT 3: Retail Business Intelligence with SQL
======================================================
Skills: SQL, SQLite, CTEs, Window Functions, Subqueries
Resume Line: "Designed and queried a normalized retail database using advanced SQL
             (CTEs, window functions, subqueries) to generate actionable BI reports"
======================================================
"""

import sqlite3
import random
import json
from datetime import datetime, timedelta
from contextlib import contextmanager


DB_PATH = "retail_analytics.db"


# ─── 1. DATABASE SETUP ───────────────────────────────────────────────────────

SCHEMA = """
-- Customers
CREATE TABLE IF NOT EXISTS customers (
    customer_id   INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    email         TEXT UNIQUE,
    city          TEXT,
    signup_date   DATE,
    tier          TEXT CHECK(tier IN ('Bronze','Silver','Gold','Platinum'))
);

-- Products
CREATE TABLE IF NOT EXISTS products (
    product_id    INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    category      TEXT,
    subcategory   TEXT,
    unit_cost     REAL,
    unit_price    REAL,
    stock_qty     INTEGER
);

-- Orders header
CREATE TABLE IF NOT EXISTS orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER REFERENCES customers(customer_id),
    order_date    DATE,
    ship_date     DATE,
    status        TEXT CHECK(status IN ('Completed','Returned','Cancelled','Pending')),
    channel       TEXT
);

-- Order line items
CREATE TABLE IF NOT EXISTS order_items (
    item_id       INTEGER PRIMARY KEY,
    order_id      INTEGER REFERENCES orders(order_id),
    product_id    INTEGER REFERENCES products(product_id),
    quantity      INTEGER,
    unit_price    REAL,
    discount      REAL DEFAULT 0
);

-- Indexes
CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id);
CREATE INDEX IF NOT EXISTS idx_orders_date     ON orders(order_date);
CREATE INDEX IF NOT EXISTS idx_items_order     ON order_items(order_id);
CREATE INDEX IF NOT EXISTS idx_items_product   ON order_items(product_id);
"""


@contextmanager
def get_conn(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def setup_database():
    """Create schema and populate with realistic data."""
    random.seed(2024)

    cities       = ["Mumbai","Delhi","Bangalore","Chennai","Hyderabad","Pune","Kolkata"]
    tiers        = ["Bronze","Silver","Gold","Platinum"]
    tier_probs   = [0.5, 0.3, 0.15, 0.05]
    categories   = {
        "Electronics": {"sub": ["Phones","Laptops","Accessories"], "cost": (200,1200), "price_mult": 1.3},
        "Clothing":    {"sub": ["Men","Women","Kids"],              "cost": (15, 80),   "price_mult": 2.5},
        "Books":       {"sub": ["Fiction","Tech","Self-Help"],      "cost": (5,  25),   "price_mult": 2.0},
        "Home":        {"sub": ["Furniture","Decor","Kitchen"],     "cost": (30, 400),  "price_mult": 1.8},
        "Sports":      {"sub": ["Gym","Outdoor","Cricket"],         "cost": (20, 300),  "price_mult": 1.9},
    }
    channels     = ["Web","Mobile","Store","Partner"]
    statuses     = ["Completed","Completed","Completed","Returned","Cancelled"]

    with get_conn() as conn:
        conn.executescript(SCHEMA)

        # Customers (500)
        for i in range(1, 501):
            tier = random.choices(tiers, tier_probs)[0]
            sd   = datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1460))
            conn.execute(
                "INSERT OR IGNORE INTO customers VALUES (?,?,?,?,?,?)",
                (i, f"Customer_{i}", f"cust{i}@mail.com",
                 random.choice(cities), sd.strftime("%Y-%m-%d"), tier)
            )

        # Products (200)
        pid = 1
        for cat, meta in categories.items():
            for sub in meta["sub"]:
                for _ in range(13):
                    cost  = round(random.uniform(*meta["cost"]), 2)
                    price = round(cost * meta["price_mult"] * random.uniform(0.9, 1.1), 2)
                    conn.execute(
                        "INSERT OR IGNORE INTO products VALUES (?,?,?,?,?,?,?)",
                        (pid, f"{sub} Product {pid}", cat, sub, cost, price,
                         random.randint(0, 500))
                    )
                    pid += 1

        # Orders + items (8,000 orders)
        start = datetime(2022, 1, 1)
        for oid in range(1, 8001):
            cust_id    = random.randint(1, 500)
            order_date = start + timedelta(days=random.randint(0, 729))
            ship_date  = order_date + timedelta(days=random.randint(1, 7))
            status     = random.choice(statuses)
            conn.execute(
                "INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?)",
                (oid, cust_id, order_date.strftime("%Y-%m-%d"),
                 ship_date.strftime("%Y-%m-%d"), status, random.choice(channels))
            )
            # 1-5 items per order
            for _ in range(random.randint(1, 5)):
                p_id  = random.randint(1, pid - 1)
                qty   = random.randint(1, 3)
                disc  = random.choice([0, 0, 0, 0.05, 0.1, 0.15])
                conn.execute(
                    "INSERT INTO order_items(order_id,product_id,quantity,unit_price,discount) "
                    "VALUES (?,?,?,?,?)",
                    (oid, p_id,
                     qty, round(random.uniform(10, 1200), 2), disc)
                )

        print("  ✓ Database created with 500 customers, 200 products, 8,000 orders")


# ─── 2. SQL QUERIES ──────────────────────────────────────────────────────────

QUERIES = {

    # ── Q1: Revenue by Category with YoY Growth (CTE + Window) ─────────────
    "revenue_by_category": """
        WITH yearly AS (
            SELECT
                p.category,
                strftime('%Y', o.order_date) AS year,
                ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2) AS revenue
            FROM order_items i
            JOIN orders   o ON i.order_id   = o.order_id
            JOIN products p ON i.product_id = p.product_id
            WHERE o.status = 'Completed'
            GROUP BY p.category, year
        ),
        with_growth AS (
            SELECT
                category, year, revenue,
                LAG(revenue) OVER (PARTITION BY category ORDER BY year) AS prev_revenue
            FROM yearly
        )
        SELECT
            category, year, revenue,
            ROUND(
                (revenue - prev_revenue) / NULLIF(prev_revenue, 0) * 100, 1
            ) AS yoy_growth_pct
        FROM with_growth
        ORDER BY category, year
    """,

    # ── Q2: Customer Lifetime Value + Ranking ───────────────────────────────
    "customer_ltv": """
        WITH customer_spend AS (
            SELECT
                c.customer_id, c.name, c.tier, c.city,
                COUNT(DISTINCT o.order_id)                                         AS total_orders,
                ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2)        AS total_spend,
                ROUND(AVG(i.quantity * i.unit_price * (1 - i.discount)), 2)        AS avg_order_value,
                MIN(o.order_date)                                                   AS first_order,
                MAX(o.order_date)                                                   AS last_order
            FROM customers c
            JOIN orders     o ON c.customer_id = o.customer_id
            JOIN order_items i ON o.order_id   = i.order_id
            WHERE o.status = 'Completed'
            GROUP BY c.customer_id
        )
        SELECT
            customer_id, name, tier, city,
            total_orders, total_spend, avg_order_value,
            first_order, last_order,
            RANK() OVER (ORDER BY total_spend DESC) AS spend_rank,
            ROUND(
                PERCENT_RANK() OVER (ORDER BY total_spend) * 100, 1
            ) AS spend_percentile
        FROM customer_spend
        ORDER BY total_spend DESC
        LIMIT 20
    """,

    # ── Q3: Product Performance with Margin ─────────────────────────────────
    "product_profitability": """
        SELECT
            p.product_id,
            p.name,
            p.category,
            p.subcategory,
            COUNT(i.item_id)                                                    AS times_sold,
            SUM(i.quantity)                                                     AS units_sold,
            ROUND(SUM(i.quantity * i.unit_price * (1-i.discount)), 2)          AS revenue,
            ROUND(SUM(i.quantity * p.unit_cost), 2)                             AS cogs,
            ROUND(SUM(i.quantity * (i.unit_price*(1-i.discount) - p.unit_cost)), 2) AS gross_profit,
            ROUND(
                SUM(i.quantity * (i.unit_price*(1-i.discount) - p.unit_cost))
                / NULLIF(SUM(i.quantity * i.unit_price*(1-i.discount)), 0) * 100,
                1
            ) AS margin_pct
        FROM products p
        JOIN order_items i ON p.product_id = i.product_id
        JOIN orders       o ON i.order_id  = o.order_id
        WHERE o.status = 'Completed'
        GROUP BY p.product_id
        ORDER BY gross_profit DESC
        LIMIT 15
    """,

    # ── Q4: Monthly Cohort Retention ─────────────────────────────────────────
    "cohort_retention": """
        WITH first_order AS (
            SELECT customer_id, MIN(strftime('%Y-%m', order_date)) AS cohort_month
            FROM orders WHERE status = 'Completed'
            GROUP BY customer_id
        ),
        activity AS (
            SELECT
                f.cohort_month,
                strftime('%Y-%m', o.order_date) AS activity_month,
                COUNT(DISTINCT o.customer_id) AS active_customers
            FROM first_order f
            JOIN orders o ON f.customer_id = o.customer_id
            WHERE o.status = 'Completed'
            GROUP BY f.cohort_month, activity_month
        ),
        cohort_size AS (
            SELECT cohort_month, COUNT(*) AS size FROM first_order GROUP BY cohort_month
        )
        SELECT
            a.cohort_month,
            a.activity_month,
            c.size AS cohort_size,
            a.active_customers,
            ROUND(a.active_customers * 100.0 / c.size, 1) AS retention_pct
        FROM activity a
        JOIN cohort_size c ON a.cohort_month = c.cohort_month
        ORDER BY a.cohort_month, a.activity_month
        LIMIT 60
    """,

    # ── Q5: Inventory & Reorder Alerts ──────────────────────────────────────
    "inventory_alerts": """
        WITH sold_30d AS (
            SELECT
                i.product_id,
                SUM(i.quantity) AS qty_sold_30d
            FROM order_items i
            JOIN orders o ON i.order_id = o.order_id
            WHERE o.order_date >= date('now', '-30 days')
              AND o.status = 'Completed'
            GROUP BY i.product_id
        )
        SELECT
            p.product_id, p.name, p.category,
            p.stock_qty,
            COALESCE(s.qty_sold_30d, 0)                          AS sold_30d,
            ROUND(COALESCE(s.qty_sold_30d, 0) / 30.0, 2)         AS daily_velocity,
            CASE
                WHEN p.stock_qty = 0 THEN 'OUT OF STOCK'
                WHEN p.stock_qty < COALESCE(s.qty_sold_30d,0)*0.5 THEN 'CRITICAL'
                WHEN p.stock_qty < COALESCE(s.qty_sold_30d,0)     THEN 'LOW'
                ELSE 'OK'
            END AS stock_status
        FROM products p
        LEFT JOIN sold_30d s ON p.product_id = s.product_id
        ORDER BY
            CASE stock_status
                WHEN 'OUT OF STOCK' THEN 1
                WHEN 'CRITICAL'     THEN 2
                WHEN 'LOW'          THEN 3
                ELSE 4
            END,
            daily_velocity DESC
        LIMIT 20
    """,

    # ── Q6: Sales Funnel by Channel ──────────────────────────────────────────
    "channel_funnel": """
        SELECT
            channel,
            COUNT(*)                                              AS total_orders,
            SUM(CASE WHEN status='Completed'  THEN 1 ELSE 0 END) AS completed,
            SUM(CASE WHEN status='Returned'   THEN 1 ELSE 0 END) AS returned,
            SUM(CASE WHEN status='Cancelled'  THEN 1 ELSE 0 END) AS cancelled,
            ROUND(SUM(CASE WHEN status='Completed' THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS completion_rate,
            ROUND(SUM(CASE WHEN status='Returned'  THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS return_rate
        FROM orders
        GROUP BY channel
        ORDER BY total_orders DESC
    """,
}


# ─── 3. QUERY RUNNER & REPORTER ──────────────────────────────────────────────

def run_query(conn, sql: str) -> list[dict]:
    """Execute SQL and return list of dicts."""
    rows = conn.execute(sql).fetchall()
    return [dict(row) for row in rows]


def print_results(title: str, rows: list[dict], max_rows=10):
    if not rows:
        print(f"  [{title}] No results")
        return
    cols = list(rows[0].keys())
    widths = {c: max(len(c), max(len(str(r.get(c, ""))) for r in rows)) for c in cols}

    print(f"\n  ┌─ {title}")
    header = "  │ " + "  ".join(c.ljust(widths[c]) for c in cols)
    print(header)
    print("  │ " + "  ".join("─" * widths[c] for c in cols))
    for row in rows[:max_rows]:
        line = "  │ " + "  ".join(str(row.get(c, "")).ljust(widths[c]) for c in cols)
        print(line)
    if len(rows) > max_rows:
        print(f"  │  ... ({len(rows) - max_rows} more rows)")
    print(f"  └{'─'*60}")


# ─── 4. MAIN ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("\n" + "█"*55)
    print("  RETAIL BUSINESS INTELLIGENCE — SQL Project")
    print("█"*55)

    print("\n[1/3] Setting up database...")
    setup_database()

    print("\n[2/3] Running SQL analyses...")
    with get_conn() as conn:
        results = {}

        print_results("Revenue by Category (YoY Growth)",
                      run_query(conn, QUERIES["revenue_by_category"]))

        print_results("Top 20 Customers by Lifetime Value",
                      run_query(conn, QUERIES["customer_ltv"]))

        print_results("Most Profitable Products",
                      run_query(conn, QUERIES["product_profitability"]))

        print_results("Cohort Retention Sample",
                      run_query(conn, QUERIES["cohort_retention"]))

        print_results("Inventory Alerts",
                      run_query(conn, QUERIES["inventory_alerts"]))

        print_results("Channel Funnel Performance",
                      run_query(conn, QUERIES["channel_funnel"]))

        for key, sql in QUERIES.items():
            results[key] = run_query(conn, sql)

    print("\n[3/3] Saving query results...")
    with open("sql_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("  ✓ sql_results.json")
    print("  ✓ retail_analytics.db")
    print("\n  Done! ✓\n")


███████████████████████████████████████████████████████
  RETAIL BUSINESS INTELLIGENCE — SQL Project
███████████████████████████████████████████████████████

[1/3] Setting up database...
  ✓ Database created with 500 customers, 200 products, 8,000 orders

[2/3] Running SQL analyses...

  ┌─ Revenue by Category (YoY Growth)
  │ category     year  revenue     yoy_growth_pct
  │ ───────────  ────  ──────────  ──────────────
  │ Books        2022  1655359.43  None          
  │ Books        2023  1648923.79  -0.4          
  │ Clothing     2022  1645508.49  None          
  │ Clothing     2023  1727700.44  5.0           
  │ Electronics  2022  1713263.28  None          
  │ Electronics  2023  1684936.66  -1.7          
  │ Home         2022  1646650.4   None          
  │ Home         2023  1666101.01  1.2           
  │ Sports       2022  1562126.89  None          
  │ Sports       2023  1583335.33  1.4           
  └────────────────────────────────────────────────────────────

  ┌─ Top 